# Sweep testing — Graph-Embedding-Based Structured CNN Pruning

Companion to `run_pipeline.ipynb`, which runs **one** `PipelineConfig` and shows its results.
This notebook runs **many** configs in one call — a `base_config` plus a grid of overrides
you define below (e.g. several `similarity_threshold` values crossed with several
`graph_embedding_method`s) — and collects every run's results into one comparison table.

Nothing here duplicates pipeline logic: every run still goes through the exact same
`run_pipeline(config)` in `src/pipeline.py`. This notebook only adds the loop, the
override-grid mechanism, and the aggregated results table around it.


## 1. Clone the repo and install dependencies

In [ ]:
!git clone https://github.com/TDShemTov/pruningconvnetspaper.git
%cd pruningconvnetspaper
!pip install -q -r requirements.txt


## 2. Base config

Same surface as `run_pipeline.ipynb`'s config cell — every field below is a real
`PipelineConfig` field, all five embedding-method configs are defined together (only
the one matching `graph_embedding_method` is ever read), see the README's "Config
reference" table for what each field does.

This `base_config` is the template every sweep run starts from — the grid in
section 3 only overrides the specific field(s) you're sweeping, everything else here
stays fixed across the whole sweep so the comparison is apples-to-apples.

Defaults below match the FashionMNIST/ResNet18 setup already explored in
`MNIST_EXPERIMENTS/` (30% prune, threshold 0.97, `n_clusters=30`), so a threshold/method
sweep here is directly comparable to those runs. To move to CIFAR-100 once you're ready,
either change `dataset_name`/`input_size` below, or add `"dataset_name"` as another sweep
axis in section 3 to run both in one call.


In [ ]:
import torch

from src.pipeline import PipelineConfig, train_base, prune_and_compare
from src.data.datasets import SplitConfig
from src.train import TrainConfig
from src.embedding import ActivationConfig
from src.graph import (
    GraphConfig,
    Node2VecConfig,
    SpectralEmbedConfig,
    RawEmbedConfig,
    DiffusionEmbedConfig,
    GCNEmbedConfig,
)
from src.clustering import ClusterConfig
from src.eval import TimingConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# --- reproducibility: True = fixed seed (same data split / weight init /
# shuffling for every run this notebook launches). False = no manual_seed
# call at all -- each model-affecting group in the sweep (see section 3)
# trains from a genuinely independent random draw, useful for checking
# whether a method ranking holds up across more than one trained model
# instead of just one lucky/unlucky training run.
fixed_split = True
fixed_seed = True
split_seed = 42 if fixed_split else None
train_seed = 42 if fixed_seed else None

node2vec_config = Node2VecConfig(embed_dim=64, walk_length=30, num_walks=50, epochs=5, device=device)
spectral_config = SpectralEmbedConfig(embed_dim=64)
raw_embed_config = RawEmbedConfig(embed_dim=64)
diffusion_config = DiffusionEmbedConfig(embed_dim=64, k=2, use_edge_weights=True)
gcn_config = GCNEmbedConfig(embed_dim=64, hidden_dim=128, epochs=100, lr=0.01, use_edge_weights=True, device=device)

base_config = PipelineConfig(
    # --- data ---
    dataset_name="fashionmnist",
    data_root="./data",
    split_config=SplitConfig(train_frac=0.8, test_frac=0.1, embed_frac=0.1, seed=split_seed),
    embed_sample_limit=None,

    # --- model ---
    model_name="resnet18",
    small_inputs=True,
    input_size=28,

    # --- baseline training ---
    baseline_train_config=TrainConfig(
        epochs=5, batch_size=128, lr=0.1, momentum=0.9, weight_decay=5e-4,
        optimizer="sgd", scheduler="cosine", device=device, seed=train_seed,
    ),

    # --- activation extraction ---
    activation_config=ActivationConfig(
        stats=["mean", "max", "std", "median", "skew", "kurtosis", "entropy"],
        batch_size=128, device=device,
    ),

    # --- similarity graph: overridden per-run by the sweep below ---
    graph_config=GraphConfig(similarity_threshold=0.97, same_layer_only=False, device=device),

    # --- graph embedding: overridden per-run by the sweep below (node2vec already
    # established as not worth its ~15-80x time overhead -- left here only so it's
    # easy to add back into a sweep axis if you ever want to re-check that) ---
    graph_embedding_method="spectral",
    node2vec_config=node2vec_config,
    spectral_config=spectral_config,
    raw_embed_config=raw_embed_config,
    diffusion_config=diffusion_config,
    gcn_config=gcn_config,

    # --- clustering ---
    cluster_config=ClusterConfig(method="ward", metric="euclidean", n_clusters=30, min_cluster_size=3),

    # --- pruning ---
    prune_fraction=0.3,
    l2_global_pruning=True,

    # --- recalibration ---
    recalibration_config=TrainConfig(
        epochs=5, batch_size=128, lr=0.005, momentum=0.9, weight_decay=5e-4,
        optimizer="sgd", scheduler="cosine", device=device, seed=train_seed,
    ),

    # --- comparison eval ---
    timing_config=TimingConfig(num_warmup=10, num_trials=30, device=device),
    eval_batch_size=128,
    seed=train_seed,

    # --- logging: separate folder from run_pipeline.ipynb's default ("experiments")
    # and from any manually-named experiment dumps, so sweeps never collide with a
    # single-run notebook's logs. Gitignored -- see repo .gitignore. ---
    log_dir="SWEEP_EXPERIMENTS",
    log_verbose=True,
)


## 3. Define the sweep

`sweep_axes` maps a config field to the list of values to try it at. A field can be
top-level (`"graph_embedding_method"`, `"dataset_name"`, `"prune_fraction"`, ...) or
nested via dotted path into any of the sub-configs (`"graph_config.similarity_threshold"`,
`"cluster_config.n_clusters"`, `"node2vec_config.p"`, ...). `build_grid` takes the
cartesian product of every axis, so N axes with sizes `a, b, c` queue `a*b*c` runs.

Not every axis costs a retrain, though: `MODEL_AFFECTING_PATHS` (below) lists the
fields that actually change the trained weights (dataset, model architecture,
`baseline_train_config.*`, `split_config.*`, the top-level `seed`). The run loop in
section 4 groups the grid by only those fields' values, calls `train_base` once per
group, and reuses that trained model for every combo in the group via
`prune_and_compare` -- so sweeping graph/embedding/clustering/pruning axes (the common
case) trains exactly once, not once per combo. Add a dataset or architecture axis and
you'll correctly get one `train_base` call per distinct value.

Default below is the connectivity check discussed earlier: does lowering the similarity
threshold (denser graph, fewer isolated filters) change which embedding method wins,
for the topology-aware methods. `raw` is deliberately excluded from the default method
list since it ignores the graph entirely — threshold is a no-op for it, already
confirmed in `MNIST_EXPERIMENTS`.


In [ ]:
import itertools


def set_by_path(obj, path, value):
    """Set `obj.a.b.c = value` given the dotted string path "a.b.c"."""
    parts = path.split(".")
    target = obj
    for p in parts[:-1]:
        target = getattr(target, p)
    setattr(target, parts[-1], value)


def build_grid(axes: dict) -> list:
    """Cartesian product of every axis, as a list of {path: value} override dicts."""
    keys = list(axes.keys())
    combos = itertools.product(*[axes[k] for k in keys])
    return [dict(zip(keys, combo)) for combo in combos]


MODEL_AFFECTING_PATHS = (
    "dataset_name", "data_root", "embed_sample_limit",
    "model_name", "small_inputs", "input_size",
    "split_config.", "baseline_train_config.", "seed",
)


def _is_model_affecting(path: str) -> bool:
    return any(path == p or path.startswith(p) for p in MODEL_AFFECTING_PATHS)


def group_by_model(grid: list) -> dict:
    """Groups grid combos that share the same model-affecting overrides, preserving
    first-seen order. Each group's combos differ only in downstream (graph/embedding/
    clustering/pruning/recalibration) fields, so they can all reuse one train_base()."""
    groups = {}
    for overrides in grid:
        key = tuple(sorted((k, v) for k, v in overrides.items() if _is_model_affecting(k)))
        groups.setdefault(key, []).append(overrides)
    return groups


sweep_axes = {
    "graph_config.similarity_threshold": [0.97, 0.90, 0.85],
    "graph_embedding_method": ["spectral", "diffusion", "gcn"],
}

grid = build_grid(sweep_axes)
print(f"{len(grid)} runs queued:")
for overrides in grid:
    print(" ", overrides)


## 4. Run the sweep

In [ ]:
import copy
import time

sweep_records = []
groups = group_by_model(grid)
print(f"{len(grid)} runs queued across {len(groups)} distinct base model(s)")

for group_overrides in groups.values():
    model_overrides = {k: v for k, v in group_overrides[0].items() if _is_model_affecting(k)}
    base_config_for_group = copy.deepcopy(base_config)
    for path, value in model_overrides.items():
        set_by_path(base_config_for_group, path, value)

    model_label = ", ".join(f"{k}={v}" for k, v in model_overrides.items()) or "base_config defaults"
    print(f"\ntraining base model for: {model_label}")
    t0 = time.time()
    base = train_base(base_config_for_group)
    print(f"  base trained in {time.time() - t0:.0f}s")

    for i, overrides in enumerate(group_overrides):
        run_config = copy.deepcopy(base_config)
        for path, value in overrides.items():
            set_by_path(run_config, path, value)

        label = ", ".join(f"{k}={v}" for k, v in overrides.items())
        print(f"  [{i + 1}/{len(group_overrides)}] running: {label}")

        t0 = time.time()
        try:
            result = prune_and_compare(base, run_config)
        except KeyboardInterrupt:
            raise
        except Exception as e:
            # A single config failing (e.g. a dense-graph OOM on gcn) shouldn't kill an
            # unattended multi-run sweep -- record it and move on to the next combo.
            print(f"    FAILED: {e}")
            sweep_records.append({**overrides, "status": "failed", "error": str(e)})
            continue

        elapsed = time.time() - t0
        print(
            f"    done in {elapsed:.0f}s -- "
            f"ours_acc={result.ours.test_metrics['accuracy']:.4f}  "
            f"l2_acc={result.l2_baseline.test_metrics['accuracy']:.4f}  "
            f"ours_params_x={result.ours_vs_baseline['params_compression']:.2f}  "
            f"ours_ops_x={result.ours_vs_baseline['ops_compression']:.2f}  "
            f"edges={result.graph_num_edges}"
        )
        sweep_records.append({**overrides, "status": "ok", "result": result})

failed = [r for r in sweep_records if r["status"] == "failed"]
print(f"\n{len(grid) - len(failed)}/{len(grid)} runs succeeded.")


## 5. Results table

One row per successful run, with the swept parameters as columns alongside the
metrics that matter for "was the overhead worth it": accuracy for all three variants,
compression/speedup ratios for ours vs. L2, graph edge count, and the graph_embedding
step's own wall-clock cost.


In [ ]:
import pandas as pd


def sweep_row(overrides, result):
    row = dict(overrides)
    row.update({
        "num_edges": result.graph_num_edges,
        "num_pruned": result.num_pruned,
        "graph_embedding_s": result.step_durations_seconds.get("graph_embedding"),
        "total_s": sum(result.step_durations_seconds.values()),
        "baseline_acc": result.baseline.test_metrics["accuracy"],
        "ours_acc": result.ours.test_metrics["accuracy"],
        "l2_acc": result.l2_baseline.test_metrics["accuracy"],
        "ours_params_x": result.ours_vs_baseline["params_compression"],
        "ours_ops_x": result.ours_vs_baseline["ops_compression"],
        "ours_speedup": result.ours_vs_baseline["speedup"],
        "l2_params_x": result.l2_vs_baseline["params_compression"],
        "l2_ops_x": result.l2_vs_baseline["ops_compression"],
        "l2_speedup": result.l2_vs_baseline["speedup"],
    })
    return row


rows = [
    sweep_row({k: v for k, v in rec.items() if k not in ("result", "status", "error")}, rec["result"])
    for rec in sweep_records
    if rec["status"] == "ok"
]
sweep_df = pd.DataFrame(rows)
sweep_df


In [ ]:
if failed:
    print(f"{len(failed)} run(s) failed:")
    for rec in failed:
        overrides = {k: v for k, v in rec.items() if k not in ("status", "error")}
        print(" ", overrides, "->", rec["error"])
else:
    print("All runs succeeded, nothing to show here.")


## 6. (Optional) Quick plot

A quick sanity-check line plot, not a polished dashboard — the default below expects
`graph_config.similarity_threshold` and `graph_embedding_method` to both be swept axes
(section 3's default). Adjust the column names here if you changed `sweep_axes`.


In [ ]:
import matplotlib.pyplot as plt

threshold_col = "graph_config.similarity_threshold"
method_col = "graph_embedding_method"

if threshold_col in sweep_df.columns and method_col in sweep_df.columns:
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for method, sub in sweep_df.groupby(method_col):
        sub = sub.sort_values(threshold_col)
        ax.plot(sub[threshold_col], sub["ours_ops_x"], marker="o", label=method)
    ax.set_xlabel("similarity_threshold (lower = denser graph)")
    ax.set_ylabel("ours ops compression (x baseline)")
    ax.set_title("Ops compression vs. graph connectivity, by embedding method")
    ax.invert_xaxis()
    ax.legend()
    plt.show()
else:
    print(
        f"Plot skipped -- expected '{threshold_col}' and '{method_col}' in sweep_axes. "
        "Adjust this cell to match whatever axes you actually swept."
    )
